## DB - PostgreSQL (Iteration 3)

Purpose: This notebook initiates the setup of our database.

As of 11/24/2025, the DMEPOS and OIG data have a concrete plan for structuring the database. This is not the final version but provides a foundation for completion.

Update (12/5/2025): The datasets are still under exploration. The database is expected to be ready before the start of the Winter break.

Update (12/16/2025): Database completion delayed. The database is expected to be ready before the start of the Spring 2026 semester. 

That being said, DMEPOS AND OIG DBs have been built and loaded with their respective data.

Note: 

Only Payments and Census data is left.

In [3]:
import pandas as pd
import psycopg2
import getpass
import numpy as np
from psycopg2.extensions import adapt, register_adapter, AsIs
from psycopg2.extras import execute_values

In [17]:
# This collects a masked password from the user
mypasswd = getpass.getpass()

········


In [245]:
connection = psycopg2.connect(
    database="casestudycf25t02",
    user="ukgff",
    host='pgsql.dsa.lan',  
    password=mypasswd
)
cursor = connection.cursor()

if connection.closed == 0:
    print("Connected to PostgreSQL!")
else:
    print("Connection failed.")
#del mypasswd

Connected to PostgreSQL!


-------------------------------------------------------------------------------------------------------

## Delete Commands

----------------------------------------------------------------------------------------------------------

## Display DBs

In [212]:
cursor.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('information_schema','pg_catalog');
""")
tables = cursor.fetchall()
for t in tables:
    print(t)

('public', 'dim_oig_exclusions')
('public', 'dim_service')
('public', 'fact_dmepos_ref_provider_and_service')
('public', 'dim_service_grouping')
('public', 'dim_npi_and_year')


------------------------------

## DB creatation and data loading

The following datasets have their databases built and data loaded.

For some datasets, the number of entries was too large, so the data had to be loaded in chunks.

Aside from this, there were no other issues, and the code executed smoothly.

Note: 

Only Payments and Census data is left.

## Dim Service

In [119]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_serv.csv")

In [120]:
df.head()

,HCPCS_CD,HCPCS_Desc
0,e0431,portable_gaseous_oxygen_system_rental_includes...
1,e1390,oxygen_concentrator_single_delivery_port_capab...
2,k0002,standard_hemi_low_seat_wheelchair
3,k0195,elevating_leg_rests_pair_for_use_with_capped_r...
4,a4253,blood_glucose_test_or_reagent_strips_for_home_...


In [121]:
df.shape

(1137, 2)

In [122]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS Dim_Service (
    HCPCS_CD VARCHAR(50),
    HCPCS_Desc VARCHAR,
    PRIMARY KEY (HCPCS_CD,HCPCS_Desc)
);
"""
cursor.execute(create_table_sql)
connection.commit()

In [123]:
sql = """
INSERT INTO Dim_Service (
    HCPCS_CD, HCPCS_Desc
) VALUES %s
"""

In [124]:
chunk_size_csv = 10000 

csv_file = "/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_serv.csv"

chunk_iter = pd.read_csv(csv_file, chunksize=chunk_size_csv)
id_n = 0

for chunk in chunk_iter:
    id_n += 1
    temp = chunk[['HCPCS_CD', 'HCPCS_Desc']].copy()    
    
    load = [tuple(row) for row in temp.to_numpy()]
    execute_values(cursor, sql, load)
    connection.commit()
    
    print(f"Chunk {id_n} inserted")

Chunk 1 inserted


In [125]:
cursor.execute(f"select count(*) from Dim_Service")
result = cursor.fetchone()
result = result[0]
print(result)

1137


In [219]:
cursor.execute(f"select * from Dim_Service;")


rows = cursor.fetchall()


print(f"Total rows: {len(rows)}\n")

for row in rows[:5]:
    print(row)

Total rows: 1137

('e0431', 'portable_gaseous_oxygen_system_rental_includes_portable_container_regulator_flowmeter_humidifier_cannula_or_mask_and_tubing')
('e1390', 'oxygen_concentrator_single_delivery_port_capable_of_delivering_85_percent_or_greater_oxygen_concentration_at_the_prescribed_flow_rate')
('k0002', 'standard_hemi_low_seat_wheelchair')
('k0195', 'elevating_leg_rests_pair_for_use_with_capped_rental_wheelchair_base')
('a4253', 'blood_glucose_test_or_reagent_strips_for_home_blood_glucose_monitor_per_50_strips')


------

## Dim Service Grouping

In [129]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_grouping.csv")

In [130]:
df.head()

,RBCS_Lvl,RBCS_Id,RBCS_Desc,HCPCS_CD,HCPCS_Desc
0,durable_medical_equipment,dc000n,dmeoxygen_and_supplies,e0431,portable_gaseous_oxygen_system_rental_includes...
1,durable_medical_equipment,dc002n,dmeoxygen_and_supplies,e1390,oxygen_concentrator_single_delivery_port_capab...
2,durable_medical_equipment,dd000n,dmewheelchairs,k0002,standard_hemi_low_seat_wheelchair
3,durable_medical_equipment,dd021n,dmewheelchairs,k0195,elevating_leg_rests_pair_for_use_with_capped_r...
4,durable_medical_equipment,de017n,dmeother_dme,a4253,blood_glucose_test_or_reagent_strips_for_home_...


In [131]:
df.shape

(1267, 5)

In [134]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS Dim_Service_Grouping (
    RBCS_Lvl VARCHAR,
    RBCS_Id VARCHAR,
    RBCS_Desc VARCHAR,
    HCPCS_CD VARCHAR(50),
    HCPCS_Desc VARCHAR,
    PRIMARY KEY (RBCS_Lvl, RBCS_Id, RBCS_Desc, HCPCS_Desc),
    FOREIGN KEY (HCPCS_CD, HCPCS_Desc) REFERENCES Dim_Service(HCPCS_CD, HCPCS_Desc)
);
"""
cursor.execute(create_table_sql)
connection.commit()

In [135]:
sql = """
INSERT INTO Dim_Service_Grouping (
    RBCS_Lvl, RBCS_Id, RBCS_Desc, HCPCS_CD, HCPCS_Desc
) VALUES %s
"""

In [136]:
chunk_size_csv = 10000 

csv_file = "/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_grouping.csv"

chunk_iter = pd.read_csv(csv_file, chunksize=chunk_size_csv)
id_n = 0

for chunk in chunk_iter:
    id_n += 1
    temp = chunk.copy()    
    
    load = [tuple(row) for row in temp.to_numpy()]
    execute_values(cursor, sql, load)
    connection.commit()
    
    print(f"Chunk {id_n} inserted")

Chunk 1 inserted


In [137]:
cursor.execute(f"select count(*) from Dim_Service_Grouping")
result = cursor.fetchone()
result = result[0]
print(result)

1267


In [220]:
cursor.execute(f"select * from Dim_Service_Grouping;")


rows = cursor.fetchall()


print(f"Total rows: {len(rows)}\n")

for row in rows[:5]:
    print(row)

Total rows: 1267

('durable_medical_equipment', 'dc000n', 'dmeoxygen_and_supplies', 'e0431', 'portable_gaseous_oxygen_system_rental_includes_portable_container_regulator_flowmeter_humidifier_cannula_or_mask_and_tubing')
('durable_medical_equipment', 'dc002n', 'dmeoxygen_and_supplies', 'e1390', 'oxygen_concentrator_single_delivery_port_capable_of_delivering_85_percent_or_greater_oxygen_concentration_at_the_prescribed_flow_rate')
('durable_medical_equipment', 'dd000n', 'dmewheelchairs', 'k0002', 'standard_hemi_low_seat_wheelchair')
('durable_medical_equipment', 'dd021n', 'dmewheelchairs', 'k0195', 'elevating_leg_rests_pair_for_use_with_capped_rental_wheelchair_base')
('durable_medical_equipment', 'de017n', 'dmeother_dme', 'a4253', 'blood_glucose_test_or_reagent_strips_for_home_blood_glucose_monitor_per_50_strips')


## DIM OIG Exclusion List

In [238]:
df = pd.read_csv("leie_with_valid_npi_clean.csv")

In [239]:
df.head()

,general,specialty,npi,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime,new_id,fraud_flag,excl_1128a1,excl_1128a2,excl_1128a3,excl_1128b5,excl_1128b6,excl_1128b7,excl_brch cia
0,IND- LIC HC SERV PRO,DENTIST,1861529091,1128a3,3/20/2024,1,1,['1128a3'],1,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0
1,IND- LIC HC SERV PRO,CHIROPRACTIC,1366544512,1128b7,8/16/2021,1,1,['1128b7'],1,AARON-OXENRIDER-nan-1976-06-07,1,0,0,0,0,0,1,0
2,IND- LIC HC SERV PRO,HEALTH CARE AIDE,1376214585,1128a1,1/20/2025,1,1,['1128a1'],1,AARON-WALTON-nan-1971-11-19,1,1,0,0,0,0,0,0
3,BUS OWNER/EXEC,COUNSELING CENTER,1326353459,1128a1,3/20/2022,1,1,['1128a1'],1,AARON-WILLIAMS-nan-1962-10-20,1,1,0,0,0,0,0,0
4,BUS OWNER/EXEC,ASSISTED LIVING FACI,1578130340,1128a1,7/20/2025,1,1,['1128a1'],1,ABDOULIE-LOWE-nan-1975-12-27,1,1,0,0,0,0,0,0


In [240]:
order = [c for c in df.columns if c != 'fraud_flag']
order

['general',
 'specialty',
 'npi',
 'excltype',
 'excldate',
 'num_exclusions_alltime',
 'num_exclusion_types_alltime',
 'list_exclusion_types_alltime',
 'num_addresses_alltime',
 'new_id',
 'excl_1128a1',
 'excl_1128a2',
 'excl_1128a3',
 'excl_1128b5',
 'excl_1128b6',
 'excl_1128b7',
 'excl_brch cia']

In [241]:
df = df.rename(columns={
    "general": "General",
    "specialty": "Specialty",
    "npi": "NPI",
    "excltype": "Excl_Type",
    "excldate": "Excl_Date",
    "num_exclusions_alltime": "Num_Exclusions_All_Time",
    "num_exclusion_types_alltime": "Num_Exclusion_Types_All_Time",
    "list_exclusion_types_alltime": "List_Exclusion_Types_All_Time",
    "num_addresses_alltime": "Num_Addresses_All_Time",
    "new_id": "New_ID",
    "fraud_flag": "Fraud_Flag",
    "excl_1128a1": "Excl_1128A1",
    "excl_1128a2": "Excl_1128A2",
    "excl_1128a3": "Excl_1128A3",
    "excl_1128b5": "Excl_1128B5",
    "excl_1128b6": "Excl_1128B6",
    "excl_1128b7": "Excl_1128B7",
    "excl_brch cia": "Excl_BRCH_CIA"
})

In [242]:
df.columns

Index(['General', 'Specialty', 'NPI', 'Excl_Type', 'Excl_Date',
       'Num_Exclusions_All_Time', 'Num_Exclusion_Types_All_Time',
       'List_Exclusion_Types_All_Time', 'Num_Addresses_All_Time', 'New_ID',
       'Fraud_Flag', 'Excl_1128A1', 'Excl_1128A2', 'Excl_1128A3',
       'Excl_1128B5', 'Excl_1128B6', 'Excl_1128B7', 'Excl_BRCH_CIA'],
      dtype='object')

In [243]:
df.shape

(1739, 18)

In [244]:
df.head()

,General,Specialty,NPI,Excl_Type,Excl_Date,Num_Exclusions_All_Time,Num_Exclusion_Types_All_Time,List_Exclusion_Types_All_Time,Num_Addresses_All_Time,New_ID,Fraud_Flag,Excl_1128A1,Excl_1128A2,Excl_1128A3,Excl_1128B5,Excl_1128B6,Excl_1128B7,Excl_BRCH_CIA
0,IND- LIC HC SERV PRO,DENTIST,1861529091,1128a3,3/20/2024,1,1,['1128a3'],1,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0
1,IND- LIC HC SERV PRO,CHIROPRACTIC,1366544512,1128b7,8/16/2021,1,1,['1128b7'],1,AARON-OXENRIDER-nan-1976-06-07,1,0,0,0,0,0,1,0
2,IND- LIC HC SERV PRO,HEALTH CARE AIDE,1376214585,1128a1,1/20/2025,1,1,['1128a1'],1,AARON-WALTON-nan-1971-11-19,1,1,0,0,0,0,0,0
3,BUS OWNER/EXEC,COUNSELING CENTER,1326353459,1128a1,3/20/2022,1,1,['1128a1'],1,AARON-WILLIAMS-nan-1962-10-20,1,1,0,0,0,0,0,0
4,BUS OWNER/EXEC,ASSISTED LIVING FACI,1578130340,1128a1,7/20/2025,1,1,['1128a1'],1,ABDOULIE-LOWE-nan-1975-12-27,1,1,0,0,0,0,0,0


In [228]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS Dim_OIG_Exclusions (
    General VARCHAR(100),
    Specialty VARCHAR(100),
    NPI BIGINT,
    Excl_Type VARCHAR(20),
    Excl_Date DATE,
    Num_Exclusions_All_Time INT,
    Num_Exclusion_Types_All_Time INT,
    List_Exclusion_Types_All_Time VARCHAR,  
    Num_Addresses_All_Time INT,
    New_ID VARCHAR(200),
    Excl_1128A1 INT,
    Excl_1128A2 INT,
    Excl_1128A3 INT,
    Excl_1128B5 INT,
    Excl_1128B6 INT,
    Excl_1128B7 INT,
    Excl_BRCH_CIA INT

);
"""
cursor.execute(create_table_sql)
connection.commit()

In [236]:
sql = """
INSERT INTO Dim_OIG_Exclusions (
    General, Specialty, NPI, Excl_Type, Excl_Date, Num_Exclusions_All_Time, Num_Exclusion_Types_All_Time, List_Exclusion_Types_All_Time, Num_Addresses_All_Time, New_ID, Excl_1128A1, Excl_1128A2, Excl_1128A3, Excl_1128B5, Excl_1128B6, Excl_1128B7, Excl_BRCH_CIA
) VALUES %s
"""

In [246]:
chunk_size_csv = 10000 

csv_file = "/dsa/groups/casestudycf25/team02/leie_with_valid_npi_clean.csv"

chunk_iter = pd.read_csv(csv_file, chunksize=chunk_size_csv)
id_n = 0

for chunk in chunk_iter:
    id_n += 1
    temp = chunk[order].copy()    
    
    load = [tuple(row) for row in temp.to_numpy()]
    execute_values(cursor, sql, load)
    connection.commit()
    
    print(f"Chunk {id_n} inserted")

Chunk 1 inserted


In [247]:
cursor.execute(f"select count(*) from Dim_OIG_Exclusions")
result = cursor.fetchone()
result = result[0]
print(result)

1739


In [248]:
cursor.execute(f"select * from Dim_OIG_Exclusions;")


rows = cursor.fetchall()


print(f"Total rows: {len(rows)}\n")

for row in rows[:5]:
    print(row)

Total rows: 1739

('IND- LIC HC SERV PRO', 'DENTIST', 1861529091, '1128a3', datetime.date(2024, 3, 20), 1, 1, "['1128a3']", 1, 'AAMIR-WAHAB-nan-1979-11-16', 0, 0, 1, 0, 0, 0, 0)
('IND- LIC HC SERV PRO', 'CHIROPRACTIC', 1366544512, '1128b7', datetime.date(2021, 8, 16), 1, 1, "['1128b7']", 1, 'AARON-OXENRIDER-nan-1976-06-07', 0, 0, 0, 0, 0, 1, 0)
('IND- LIC HC SERV PRO', 'HEALTH CARE AIDE', 1376214585, '1128a1', datetime.date(2025, 1, 20), 1, 1, "['1128a1']", 1, 'AARON-WALTON-nan-1971-11-19', 1, 0, 0, 0, 0, 0, 0)
('BUS OWNER/EXEC', 'COUNSELING CENTER', 1326353459, '1128a1', datetime.date(2022, 3, 20), 1, 1, "['1128a1']", 1, 'AARON-WILLIAMS-nan-1962-10-20', 1, 0, 0, 0, 0, 0, 0)
('BUS OWNER/EXEC', 'ASSISTED LIVING FACI', 1578130340, '1128a1', datetime.date(2025, 7, 20), 1, 1, "['1128a1']", 1, 'ABDOULIE-LOWE-nan-1975-12-27', 1, 0, 0, 0, 0, 0, 0)


------------------------

## DIM NPI And Year

In [173]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean.csv", nrows= 1000)

In [174]:
df.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre,Year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0,1.942167,2021
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,2.278200,2021
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,0.0,1.000000,0.846154,0.0,0.0,0.0,0.0,0.0,1.872308,2021
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,...,0.0,0.833333,0.777778,0.0,0.0,0.0,0.0,0.0,2.144684,2021
4,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,2.006500,2021


In [175]:
df.columns

Index(['Rfrg_NPI', 'Rfrg_Prvdr_Last_Name_Org', 'Rfrg_Prvdr_First_Name',
       'Rfrg_Prvdr_MI', 'Rfrg_Prvdr_Crdntls', 'Rfrg_Prvdr_Ent_Cd',
       'Rfrg_Prvdr_St1', 'Rfrg_Prvdr_St2', 'Rfrg_Prvdr_City',
       'Rfrg_Prvdr_State_Abrvtn', 'Rfrg_Prvdr_State_FIPS', 'Rfrg_Prvdr_Zip5',
       'Rfrg_Prvdr_RUCA', 'Rfrg_Prvdr_RUCA_Desc', 'Rfrg_Prvdr_Cntry',
       'Rfrg_Prvdr_Spclty_Desc', 'Rfrg_Prvdr_Spclty_Srce', 'Tot_Suplrs',
       'Tot_Suplr_HCPCS_Cds', 'Tot_Suplr_Benes', 'Tot_Suplr_Clms',
       'Tot_Suplr_Srvcs', 'Suplr_Sbmtd_Chrgs', 'Suplr_Mdcr_Alowd_Amt',
       'Suplr_Mdcr_Pymt_Amt', 'Suplr_Mdcr_Stdzd_Pymt_Amt', 'DME_Sprsn_Ind',
       'DME_Tot_Suplrs', 'DME_Tot_Suplr_HCPCS_Cds', 'DME_Tot_Suplr_Benes',
       'DME_Tot_Suplr_Clms', 'DME_Tot_Suplr_Srvcs', 'DME_Suplr_Sbmtd_Chrgs',
       'DME_Suplr_Mdcr_Alowd_Amt', 'DME_Suplr_Mdcr_Pymt_Amt',
       'DME_Suplr_Mdcr_Stdzd_Pymt_Amt', 'POS_Sprsn_Ind', 'POS_Tot_Suplrs',
       'POS_Tot_Suplr_HCPCS_Cds', 'POS_Tot_Suplr_Benes', 'POS_Tot_Suplr_Clm

In [176]:
order = ['Year', 'Rfrg_NPI', 'Rfrg_Prvdr_Spclty_Desc', 'Bene_Avg_Age', 'Bene_Age_LT_65_Cnt',
       'Bene_Age_65_74_Cnt', 'Bene_Age_75_84_Cnt', 'Bene_Age_GT_84_Cnt',
       'Bene_Feml_Cnt', 'Bene_Male_Cnt', 'Bene_Race_Wht_Cnt',
       'Bene_Race_Black_Cnt', 'Bene_Race_Api_Cnt', 'Bene_Race_Hspnc_Cnt',
       'Bene_Race_Natind_Cnt', 'Bene_Race_Othr_Cnt', 'Bene_Ndual_Cnt',
       'Bene_Dual_Cnt', 'Bene_CC_BH_ADHD_OthCD_V1_Pct',
       'Bene_CC_BH_Alcohol_Drug_V1_Pct', 'Bene_CC_BH_Tobacco_V1_Pct',
       'Bene_CC_BH_Alz_NonAlzdem_V2_Pct', 'Bene_CC_BH_Anxiety_V1_Pct',
       'Bene_CC_BH_Bipolar_V1_Pct', 'Bene_CC_BH_Mood_V2_Pct',
       'Bene_CC_BH_Depress_V1_Pct', 'Bene_CC_BH_PD_V1_Pct',
       'Bene_CC_BH_PTSD_V1_Pct', 'Bene_CC_BH_Schizo_OthPsy_V1_Pct',
       'Bene_CC_PH_Asthma_V2_Pct', 'Bene_CC_PH_Afib_V2_Pct',
       'Bene_CC_PH_Cancer6_V2_Pct', 'Bene_CC_PH_CKD_V2_Pct',
       'Bene_CC_PH_COPD_V2_Pct', 'Bene_CC_PH_Diabetes_V2_Pct',
       'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_Hyperlipidemia_V2_Pct',
       'Bene_CC_PH_Hypertension_V2_Pct', 'Bene_CC_PH_IschemicHeart_V2_Pct',
       'Bene_CC_PH_Osteoporosis_V2_Pct', 'Bene_CC_PH_Parkinson_V2_Pct',
       'Bene_CC_PH_Arthritis_V2_Pct', 'Bene_CC_PH_Stroke_TIA_V2_Pct',
       'Bene_Avg_Risk_Scre']

df = df[order]

In [177]:
df.head()

,Year,Rfrg_NPI,Rfrg_Prvdr_Spclty_Desc,Bene_Avg_Age,Bene_Age_LT_65_Cnt,Bene_Age_65_74_Cnt,Bene_Age_75_84_Cnt,Bene_Age_GT_84_Cnt,Bene_Feml_Cnt,Bene_Male_Cnt,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,2021,1003000126,Internal Medicine,79.750000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0,1.942167
1,2021,1003000480,General Surgery,68.200000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,2.278200
2,2021,1003000522,Family Practice,79.923077,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,1.000000,0.846154,0.0,0.0,0.0,0.0,0.0,1.872308
3,2021,1003000530,Internal Medicine,74.666667,0.0,0.0,0.0,0.0,0.0,0.0,...,0.722222,0.0,0.833333,0.777778,0.0,0.0,0.0,0.0,0.0,2.144684
4,2021,1003000597,Urology,70.250000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,2.006500


In [178]:
df.columns

Index(['Year', 'Rfrg_NPI', 'Rfrg_Prvdr_Spclty_Desc', 'Bene_Avg_Age',
       'Bene_Age_LT_65_Cnt', 'Bene_Age_65_74_Cnt', 'Bene_Age_75_84_Cnt',
       'Bene_Age_GT_84_Cnt', 'Bene_Feml_Cnt', 'Bene_Male_Cnt',
       'Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_Api_Cnt',
       'Bene_Race_Hspnc_Cnt', 'Bene_Race_Natind_Cnt', 'Bene_Race_Othr_Cnt',
       'Bene_Ndual_Cnt', 'Bene_Dual_Cnt', 'Bene_CC_BH_ADHD_OthCD_V1_Pct',
       'Bene_CC_BH_Alcohol_Drug_V1_Pct', 'Bene_CC_BH_Tobacco_V1_Pct',
       'Bene_CC_BH_Alz_NonAlzdem_V2_Pct', 'Bene_CC_BH_Anxiety_V1_Pct',
       'Bene_CC_BH_Bipolar_V1_Pct', 'Bene_CC_BH_Mood_V2_Pct',
       'Bene_CC_BH_Depress_V1_Pct', 'Bene_CC_BH_PD_V1_Pct',
       'Bene_CC_BH_PTSD_V1_Pct', 'Bene_CC_BH_Schizo_OthPsy_V1_Pct',
       'Bene_CC_PH_Asthma_V2_Pct', 'Bene_CC_PH_Afib_V2_Pct',
       'Bene_CC_PH_Cancer6_V2_Pct', 'Bene_CC_PH_CKD_V2_Pct',
       'Bene_CC_PH_COPD_V2_Pct', 'Bene_CC_PH_Diabetes_V2_Pct',
       'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_Hyperli

In [179]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS Dim_NPI_and_Year (
    Year INT,
    Rfrg_NPI BIGINT,
    Rfrg_Prvdr_Spclty_Desc VARCHAR(200),
    Bene_Avg_Age FLOAT,
    Bene_Age_LT_65_Cnt FLOAT,
    Bene_Age_65_74_Cnt FLOAT,
    Bene_Age_75_84_Cnt FLOAT,
    Bene_Age_GT_84_Cnt FLOAT,
    Bene_Feml_Cnt FLOAT,
    Bene_Male_Cnt FLOAT,
    Bene_Race_Wht_Cnt FLOAT,
    Bene_Race_Black_Cnt FLOAT,
    Bene_Race_Api_Cnt FLOAT,
    Bene_Race_Hspnc_Cnt FLOAT,
    Bene_Race_Natind_Cnt FLOAT,
    Bene_Race_Othr_Cnt FLOAT,
    Bene_Ndual_Cnt FLOAT,
    Bene_Dual_Cnt FLOAT,
    Bene_CC_BH_ADHD_OthCD_V1_Pct FLOAT,
    Bene_CC_BH_Alcohol_Drug_V1_Pct FLOAT,
    Bene_CC_BH_Tobacco_V1_Pct FLOAT,
    Bene_CC_BH_Alz_NonAlzdem_V2_Pct FLOAT,
    Bene_CC_BH_Anxiety_V1_Pct FLOAT,
    Bene_CC_BH_Bipolar_V1_Pct FLOAT,
    Bene_CC_BH_Mood_V2_Pct FLOAT,
    Bene_CC_BH_Depress_V1_Pct FLOAT,
    Bene_CC_BH_PD_V1_Pct FLOAT,
    Bene_CC_BH_PTSD_V1_Pct FLOAT,
    Bene_CC_BH_Schizo_OthPsy_V1_Pct FLOAT,
    Bene_CC_PH_Asthma_V2_Pct FLOAT,
    Bene_CC_PH_Afib_V2_Pct FLOAT,
    Bene_CC_PH_Cancer6_V2_Pct FLOAT,
    Bene_CC_PH_CKD_V2_Pct FLOAT,
    Bene_CC_PH_COPD_V2_Pct FLOAT,
    Bene_CC_PH_Diabetes_V2_Pct FLOAT,
    Bene_CC_PH_HF_NonIHD_V2_Pct FLOAT,
    Bene_CC_PH_Hyperlipidemia_V2_Pct FLOAT,
    Bene_CC_PH_Hypertension_V2_Pct FLOAT,
    Bene_CC_PH_IschemicHeart_V2_Pct FLOAT,
    Bene_CC_PH_Osteoporosis_V2_Pct FLOAT,
    Bene_CC_PH_Parkinson_V2_Pct FLOAT,
    Bene_CC_PH_Arthritis_V2_Pct FLOAT,
    Bene_CC_PH_Stroke_TIA_V2_Pct FLOAT,
    Bene_Avg_Risk_Scre FLOAT
);
"""
cursor.execute(create_table_sql)
connection.commit()

In [180]:
sql = """
INSERT INTO Dim_NPI_and_Year (
    Year,
    Rfrg_NPI,
    Rfrg_Prvdr_Spclty_Desc,
    Bene_Avg_Age,
    Bene_Age_LT_65_Cnt,
    Bene_Age_65_74_Cnt,
    Bene_Age_75_84_Cnt,
    Bene_Age_GT_84_Cnt,
    Bene_Feml_Cnt,
    Bene_Male_Cnt,
    Bene_Race_Wht_Cnt,
    Bene_Race_Black_Cnt,
    Bene_Race_Api_Cnt,
    Bene_Race_Hspnc_Cnt,
    Bene_Race_Natind_Cnt,
    Bene_Race_Othr_Cnt,
    Bene_Ndual_Cnt,
    Bene_Dual_Cnt,
    Bene_CC_BH_ADHD_OthCD_V1_Pct,
    Bene_CC_BH_Alcohol_Drug_V1_Pct,
    Bene_CC_BH_Tobacco_V1_Pct,
    Bene_CC_BH_Alz_NonAlzdem_V2_Pct,
    Bene_CC_BH_Anxiety_V1_Pct,
    Bene_CC_BH_Bipolar_V1_Pct,
    Bene_CC_BH_Mood_V2_Pct,
    Bene_CC_BH_Depress_V1_Pct,
    Bene_CC_BH_PD_V1_Pct,
    Bene_CC_BH_PTSD_V1_Pct,
    Bene_CC_BH_Schizo_OthPsy_V1_Pct,
    Bene_CC_PH_Asthma_V2_Pct,
    Bene_CC_PH_Afib_V2_Pct,
    Bene_CC_PH_Cancer6_V2_Pct,
    Bene_CC_PH_CKD_V2_Pct,
    Bene_CC_PH_COPD_V2_Pct,
    Bene_CC_PH_Diabetes_V2_Pct,
    Bene_CC_PH_HF_NonIHD_V2_Pct,
    Bene_CC_PH_Hyperlipidemia_V2_Pct,
    Bene_CC_PH_Hypertension_V2_Pct,
    Bene_CC_PH_IschemicHeart_V2_Pct,
    Bene_CC_PH_Osteoporosis_V2_Pct,
    Bene_CC_PH_Parkinson_V2_Pct,
    Bene_CC_PH_Arthritis_V2_Pct,
    Bene_CC_PH_Stroke_TIA_V2_Pct,
    Bene_Avg_Risk_Scre
) VALUES %s
"""

In [181]:
order = ['Year', 'Rfrg_NPI', 'Rfrg_Prvdr_Spclty_Desc', 'Bene_Avg_Age', 'Bene_Age_LT_65_Cnt',
       'Bene_Age_65_74_Cnt', 'Bene_Age_75_84_Cnt', 'Bene_Age_GT_84_Cnt',
       'Bene_Feml_Cnt', 'Bene_Male_Cnt', 'Bene_Race_Wht_Cnt',
       'Bene_Race_Black_Cnt', 'Bene_Race_Api_Cnt', 'Bene_Race_Hspnc_Cnt',
       'Bene_Race_Natind_Cnt', 'Bene_Race_Othr_Cnt', 'Bene_Ndual_Cnt',
       'Bene_Dual_Cnt', 'Bene_CC_BH_ADHD_OthCD_V1_Pct',
       'Bene_CC_BH_Alcohol_Drug_V1_Pct', 'Bene_CC_BH_Tobacco_V1_Pct',
       'Bene_CC_BH_Alz_NonAlzdem_V2_Pct', 'Bene_CC_BH_Anxiety_V1_Pct',
       'Bene_CC_BH_Bipolar_V1_Pct', 'Bene_CC_BH_Mood_V2_Pct',
       'Bene_CC_BH_Depress_V1_Pct', 'Bene_CC_BH_PD_V1_Pct',
       'Bene_CC_BH_PTSD_V1_Pct', 'Bene_CC_BH_Schizo_OthPsy_V1_Pct',
       'Bene_CC_PH_Asthma_V2_Pct', 'Bene_CC_PH_Afib_V2_Pct',
       'Bene_CC_PH_Cancer6_V2_Pct', 'Bene_CC_PH_CKD_V2_Pct',
       'Bene_CC_PH_COPD_V2_Pct', 'Bene_CC_PH_Diabetes_V2_Pct',
       'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_Hyperlipidemia_V2_Pct',
       'Bene_CC_PH_Hypertension_V2_Pct', 'Bene_CC_PH_IschemicHeart_V2_Pct',
       'Bene_CC_PH_Osteoporosis_V2_Pct', 'Bene_CC_PH_Parkinson_V2_Pct',
       'Bene_CC_PH_Arthritis_V2_Pct', 'Bene_CC_PH_Stroke_TIA_V2_Pct',
       'Bene_Avg_Risk_Scre']

chunk_size_csv = 100000 

csv_file = "/dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean.csv"

chunk_iter = pd.read_csv(csv_file, chunksize=chunk_size_csv)
id_n = 0

for chunk in chunk_iter:
    id_n += 1
    temp = chunk[order].copy()    
    
    load = [tuple(row) for row in temp.to_numpy()]
    execute_values(cursor, sql, load)
    connection.commit()
    
    print(f"Chunk {id_n} inserted")

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Chunk 1 inserted


/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Chunk 2 inserted
Chunk 3 inserted
Chunk 4 inserted
Chunk 5 inserted
Chunk 6 inserted
Chunk 7 inserted
Chunk 8 inserted
Chunk 9 inserted
Chunk 10 inserted
Chunk 11 inserted
Chunk 12 inserted


In [182]:
cursor.execute(f"select count(*) from Dim_NPI_and_Year")
result = cursor.fetchone()
result = result[0]
print(result)

1157253


In [183]:
cursor.execute(f"select * from Dim_NPI_and_Year LIMIT 5;")


rows = cursor.fetchall()


print(f"Total rows: {len(rows)}\n")

for row in rows[:5]:
    print(row)

Total rows: 5

(2021, 1003000126, 'Internal Medicine', 79.75, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.9421666667)
(2021, 1003000480, 'General Surgery', 68.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2782)
(2021, 1003000522, 'Family Practice', 79.923076923, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 13.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.8461538462, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8723076923)
(2021, 1003000530, 'Internal Medicine', 74.666666667, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 18.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7222

------------------

## DMEPOS Referal Provider and Service

In [190]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_RefPro_Ser_nar_clean.csv", nrows = 1000)

In [191]:
df.head()

,Year,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,...,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt,Excluded,Charge_Diff,Supplier
0,2021,1003000126,enkeshafi,ardalan,NaN,md,i,6410_rockledge_dr_ste_304,NaN,bethesda,...,NaN,16,16,46.336250,20.097500,14.857500,15.280000,n,26.238750,n
1,2021,1003000126,enkeshafi,ardalan,NaN,md,i,6410_rockledge_dr_ste_304,NaN,bethesda,...,NaN,19,19,360.770000,98.223158,72.843684,79.753158,n,262.546842,n
2,2021,1003000126,enkeshafi,ardalan,NaN,md,i,6410_rockledge_dr_ste_304,NaN,bethesda,...,NaN,11,11,92.000000,39.230000,31.385455,33.552727,n,52.770000,n
3,2021,1003000126,enkeshafi,ardalan,NaN,md,i,6410_rockledge_dr_ste_304,NaN,bethesda,...,NaN,11,11,20.000000,10.210909,8.169091,8.456364,n,9.789091,n
4,2021,1003000480,rothchild,kevin,b,md,i,12605_e_16th_ave,NaN,aurora,...,NaN,11,13,272.003846,80.513846,64.407692,84.701538,n,191.490000,n


In [192]:
df.columns

Index(['Year', 'Rfrg_NPI', 'Rfrg_Prvdr_Last_Name_Org', 'Rfrg_Prvdr_First_Name',
       'Rfrg_Prvdr_MI', 'Rfrg_Prvdr_Crdntls', 'Rfrg_Prvdr_Ent_Cd',
       'Rfrg_Prvdr_St1', 'Rfrg_Prvdr_St2', 'Rfrg_Prvdr_City',
       'Rfrg_Prvdr_State_Abrvtn', 'Rfrg_Prvdr_State_FIPS', 'Rfrg_Prvdr_Zip5',
       'Rfrg_Prvdr_Cntry', 'Rfrg_Prvdr_Spclty_Cd', 'Rfrg_Prvdr_Spclty_Desc',
       'Rfrg_Prvdr_Spclty_Srce', 'RBCS_Lvl', 'RBCS_Id', 'RBCS_Desc',
       'HCPCS_CD', 'HCPCS_Desc', 'Suplr_Rentl_Ind', 'Tot_Suplrs',
       'DME_Sprsn_Ind', 'Tot_Suplr_Benes', 'Tot_Suplr_Clms', 'Tot_Suplr_Srvcs',
       'Avg_Suplr_Sbmtd_Chrg', 'Avg_Suplr_Mdcr_Alowd_Amt',
       'Avg_Suplr_Mdcr_Pymt_Amt', 'Avg_Suplr_Mdcr_Stdzd_Amt', 'Excluded',
       'Charge_Diff', 'Supplier'],
      dtype='object')

In [7]:
df.shape

(4413118, 35)

In [195]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS Fact_DMEPOS_Ref_Provider_and_Service (
    Year INT,
    Rfrg_NPI BIGINT,
    Zipcode VARCHAR ,
    RBCS_Lvl VARCHAR,
    RBCS_Id VARCHAR,
    RBCS_Desc VARCHAR,
    HCPCS_CD VARCHAR(50),
    HCPCS_Desc VARCHAR,
    Suplr_Rentl_Ind CHAR(1),
    Tot_Suplrs INT,
    DME_Sprsn_Ind CHAR(1),
    Tot_Suplr_Benes FLOAT,
    Tot_Suplr_Clms INT,
    Tot_Suplr_Srvcs INT,
    Avg_Suplr_Sbmtd_Chrg FLOAT,
    Avg_Suplr_Mdcr_Alowd_Amt FLOAT,
    Avg_Suplr_Mdcr_Pymt_Amt FLOAT,
    Avg_Suplr_Mdcr_Stdzd_Amt FLOAT,
    Excluded CHAR(1),
    Supplier_indicator CHAR(1));
"""
cursor.execute(create_table_sql)
connection.commit()

In [196]:
sql = """
INSERT INTO Fact_DMEPOS_Ref_Provider_and_Service (
    Year, Rfrg_NPI, Zipcode, RBCS_Lvl, RBCS_Id, RBCS_Desc, HCPCS_CD, HCPCS_Desc, Suplr_Rentl_Ind,
    Tot_Suplrs, DME_Sprsn_Ind, Tot_Suplr_Benes, Tot_Suplr_Clms,
    Tot_Suplr_Srvcs, Avg_Suplr_Sbmtd_Chrg, Avg_Suplr_Mdcr_Alowd_Amt,
    Avg_Suplr_Mdcr_Pymt_Amt, Avg_Suplr_Mdcr_Stdzd_Amt,
    Excluded, Supplier_indicator
) VALUES %s
"""

In [197]:
chunk_size_csv = 100000 

csv_file = "/dsa/groups/casestudycf25/team02/DMEPOS_RefPro_Ser_clean_02.csv"

chunk_iter = pd.read_csv(csv_file, chunksize=chunk_size_csv)
id_n = 0

for chunk in chunk_iter:
    id_n += 1
    temp = chunk[['Year', 'Rfrg_NPI', 'Rfrg_Prvdr_Zip5', 'RBCS_Lvl', 'RBCS_Id', 'RBCS_Desc',
                  'HCPCS_CD', 'HCPCS_Desc', 'Suplr_Rentl_Ind',
                  'Tot_Suplrs', 'DME_Sprsn_Ind', 'Tot_Suplr_Benes', 'Tot_Suplr_Clms',
                  'Tot_Suplr_Srvcs', 'Avg_Suplr_Sbmtd_Chrg', 'Avg_Suplr_Mdcr_Alowd_Amt',
                  'Avg_Suplr_Mdcr_Pymt_Amt', 'Avg_Suplr_Mdcr_Stdzd_Amt',
                  'Excluded', 'Supplier']].copy()

    temp.rename(columns={'Rfrg_Prvdr_Zip5': 'Zipcode','Supplier': 'Supplier_indicator'}, inplace=True)
    
    
    load = [tuple(row) for row in temp.to_numpy()]
    execute_values(cursor, sql, load)
    connection.commit()
    
    print(f"Chunk {id_n} inserted")

Chunk 1 inserted
Chunk 2 inserted
Chunk 3 inserted
Chunk 4 inserted
Chunk 5 inserted
Chunk 6 inserted
Chunk 7 inserted
Chunk 8 inserted
Chunk 9 inserted
Chunk 10 inserted
Chunk 11 inserted
Chunk 12 inserted
Chunk 13 inserted
Chunk 14 inserted
Chunk 15 inserted
Chunk 16 inserted
Chunk 17 inserted
Chunk 18 inserted
Chunk 19 inserted
Chunk 20 inserted
Chunk 21 inserted
Chunk 22 inserted
Chunk 23 inserted
Chunk 24 inserted
Chunk 25 inserted
Chunk 26 inserted
Chunk 27 inserted
Chunk 28 inserted
Chunk 29 inserted
Chunk 30 inserted
Chunk 31 inserted
Chunk 32 inserted
Chunk 33 inserted
Chunk 34 inserted
Chunk 35 inserted
Chunk 36 inserted
Chunk 37 inserted
Chunk 38 inserted
Chunk 39 inserted
Chunk 40 inserted
Chunk 41 inserted
Chunk 42 inserted
Chunk 43 inserted
Chunk 44 inserted
Chunk 45 inserted


In [62]:
cursor.execute(f"select count(*) from Fact_DMEPOS_Ref_Provider_and_Service")
result = cursor.fetchone()
result = result[0]
print(result)

4413118


In [184]:
cursor.execute(f"select * from Fact_DMEPOS_Ref_Provider_and_Service LIMIT 5;")


rows = cursor.fetchall()


print(f"Total rows: {len(rows)}\n")

for row in rows[:5]:
    print(row)

Total rows: 5

(2021, 1003000126, '20817', 'e0431     ', 'y', 5, 'y', nan, 16, 16, 46.33625, 20.0975, 14.8575, 15.28, 'n', 'n')
(2021, 1003000126, '20817', 'e1390     ', 'y', 6, 'y', nan, 19, 19, 360.77, 98.223157895, 72.843684211, 79.753157895, 'n', 'n')
(2021, 1003000126, '20817', 'k0002     ', 'y', 1, 'y', nan, 11, 11, 92.0, 39.23, 31.385454545, 33.552727273, 'n', 'n')
(2021, 1003000126, '20817', 'k0195     ', 'y', 1, 'y', nan, 11, 11, 20.0, 10.210909091, 8.1690909091, 8.4563636364, 'n', 'n')
(2021, 1003000480, '80045', 'e1390     ', 'y', 4, 'y', nan, 11, 13, 272.00384615, 80.513846154, 64.407692308, 84.701538462, 'n', 'n')
